# Bayesian Mixed-Frequency VAR (Pipeline B)

**Model**: BVAR via `mfbvar::estimate_mfbvar`
**Target**: `gdpc1`
**Variables**: Cat3 (53, COVID dummies excluded). Quarterly vars ordered last (mfbvar requirement).
**Train**: 1959-2007 / **Val**: 2008-2016 / **Test**: 2017-2025
**Scaling**: YES (Minnesota prior needs standardized data)
**HP tuning**: NO (Minnesota lambda1=0.1, literature default)
**Prior**: Minnesota prior, n_lags=4, n_reps=100
**Note**: mfbvar requires quarterly variables at END of list. Detection from meta_data.csv freq column.
Original had 24 hardcoded vars -- ours is PROGRAMMATIC from Cat3 list.
COVID dummies excluded: zero variance in 1959-2019 training window crashes `set_prior()`. Same treatment as DFM.

In [ ]:
.libPaths(c("C:/Users/asus/R/library", "C:/Users/asus/AppData/Local/R/win-library/4.6", .libPaths()))

library(Rmisc)
library(tidyverse)
library(mfbvar)
library(imputeTS)

source("../data/helpers.R")

metadata <- read_csv("../data/meta_data.csv", show_col_types = FALSE)

# Reduced BVAR predictor set.
# Full Cat3 (53 predictors) is computationally infeasible in mfbvar on this
# setup. Use the Lasso 80% cumulative-importance cutoff documented in
# data/variable_rankings_by_rule.txt: 15 predictors plus the target.
bvar_features <- c(
  "outbs", "gcec1", "houstne", "outnfb", "hwiuratio",
  "compapff", "ulcnfb", "andenox", "mortg10yrx", "housts",
  "busloans", "ces2000000008", "amdmuox", "ces9091000001",
  "ces1021000001"
)
bvar_features <- tolower(bvar_features)

# Build ordered variable list: monthly first, quarterly last (mfbvar requirement)
var_freqs <- metadata %>%
    dplyr::filter(series %in% bvar_features) %>%
    dplyr::select(series, freq)

monthly_vars   <- var_freqs %>% dplyr::filter(freq == "m") %>% dplyr::pull(series)
quarterly_vars <- var_freqs %>% dplyr::filter(freq == "q") %>% dplyr::pull(series)

ordered_vars <- c(monthly_vars, quarterly_vars[quarterly_vars != "gdpc1"], "gdpc1")
cat("BVAR vars:", length(ordered_vars), "(", length(monthly_vars), "monthly,",
    length(quarterly_vars), "quarterly)\n")

data <- read_csv("../data/data_tf_monthly.csv", show_col_types = FALSE) %>% dplyr::arrange(date)
target_variable <- "gdpc1"

train_start_date <- "1959-01-01"
test_start_date  <- "2017-01-01"
test_end_date    <- "2025-12-01"

test <- data %>%
    dplyr::filter(date >= as.Date(train_start_date), date <= as.Date(test_end_date)) %>%
    data.frame()

for (col in colnames(test)) {
    if (is.numeric(test[, col]) && sum(is.infinite(test[, col])) > 0) {
        test[is.infinite(test[, col]), col] <- NA
    }
}

dates <- seq(as.Date(test_start_date), as.Date(test_end_date), by = "month")
dates <- dates[substr(dates, 6, 7) %in% c("03", "06", "09", "12")]

vintage_offsets <- c(m1 = -2, m2 = -1, m3 = 0)
pred_dict <- data.frame(date = dates)
for (lag_name in names(vintage_offsets)) pred_dict[, lag_name] <- NA

for (idx in seq_along(dates)) {
    date <- as.character(dates[idx])

    for (lag_name in names(vintage_offsets)) {
        cat("BVAR fit", idx, "/", length(dates), lag_name, date, "\n")
        vintage_date <- shift_month(date, vintage_offsets[[lag_name]])
        lagged_data <- gen_vintage_data(metadata, test, date, vintage_date)
        lagged_data <- data.frame(lagged_data)
        lagged_data[lagged_data$date == date, target_variable] <- NA

        n_rows <- nrow(lagged_data)
        fill_rows <- round(n_rows * 0.9)
        lagged_data[1:fill_rows, ] <- na_mean(lagged_data[1:fill_rows, ])

        # CRITICAL: filter on Cat3+gdpc1 columns only.
        # data_tf_monthly has 300 vars; non-Cat3 vars with data at the vintage fringe
        # prevent the all-NA rowSums filter from dropping those fringe rows.
        # If we keep them, the Cat3-only mf_test ts end with trailing NAs -> set_prior fails.
        avail_vars <- ordered_vars[ordered_vars %in% colnames(lagged_data)]
        lagged_cat3 <- lagged_data[, c("date", avail_vars), drop = FALSE]
        num_cols_cat3 <- length(avail_vars)
        lagged_cat3 <- lagged_cat3[
            rowSums(is.na(lagged_cat3[, -1, drop = FALSE])) < num_cols_cat3,
        ]

        # Build mfbvar-compatible list (monthly first, quarterly last)
        mf_test <- list()
        for (col in avail_vars) {
            freq_str <- metadata %>% dplyr::filter(series == !!col) %>% dplyr::pull(freq)
            if (length(freq_str) == 0) next

            if (freq_str == "q") {
                tmp_series <- lagged_cat3 %>%
                    dplyr::filter(substr(date, 6, 7) %in% c("03", "06", "09", "12")) %>%
                    dplyr::select(all_of(col)) %>% dplyr::slice(2:n()) %>% dplyr::pull()
                tmp_ts <- ts(tmp_series, start = c(1959, 2), frequency = 4)
            } else {
                tmp_series <- lagged_cat3 %>%
                    dplyr::select(all_of(col)) %>% dplyr::slice(2:n()) %>% dplyr::pull()
                tmp_ts <- ts(tmp_series, start = c(1959, 2), frequency = 12)
            }
            mf_test[[col]] <- tmp_ts
        }

        pred <- tryCatch({
            # block_exo=integer(0): R 4.6.0 changed is.atomic(NULL)=FALSE, breaking mfbvar 0.5.6
            prior <- set_prior(Y = mf_test, n_lags = 4, n_reps = 20, block_exo = integer(0))
            c_interval <- t(sapply(mf_test, Rmisc::CI, ci = 0.95))
            prior_intervals <- c_interval[, c("upper", "lower")]
            moments <- interval_to_moments(prior_intervals)
            prior <- update_prior(prior, d = "intercept",
                                  prior_psi_mean  = moments$prior_psi_mean,
                                  prior_psi_Omega = moments$prior_psi_Omega)
            prior <- update_prior(prior, n_fcst = 12)
            model <- estimate_mfbvar(prior, prior = "minn", variance = "iw")
            predict(model, pred_bands = NULL) %>%
                dplyr::filter(variable == target_variable, fcst_date == date) %>%
                dplyr::select(fcst) %>% dplyr::pull() %>% mean()
        }, error = function(e) {
            cat("ERROR at", date, lag_name, ":", conditionMessage(e), "\n")
            cat("Trying Cat2 fallback at", date, lag_name, "\n")
            tryCatch({
                # mfbvar requires weekly-monthly-quarterly ordering. This fallback has no weekly variables.
                fallback_vars <- c("houstne", "outbs", "outnfb", "gcec1", target_variable)
                fallback_vars <- fallback_vars[fallback_vars %in% colnames(lagged_data)]
                fallback_data <- lagged_data[, c("date", fallback_vars), drop = FALSE]
                fallback_data <- fallback_data[
                    rowSums(is.na(fallback_data[, -1, drop = FALSE])) < length(fallback_vars),
                ]

                fallback_mf <- list()
                for (fcol in fallback_vars) {
                    ffreq <- metadata %>% dplyr::filter(series == !!fcol) %>% dplyr::pull(freq)
                    if (ffreq == "q") {
                        fseries <- fallback_data %>%
                            dplyr::filter(substr(date, 6, 7) %in% c("03", "06", "09", "12")) %>%
                            dplyr::select(all_of(fcol)) %>% dplyr::slice(2:n()) %>% dplyr::pull()
                        fallback_mf[[fcol]] <- ts(fseries, start = c(1959, 2), frequency = 4)
                    } else {
                        fseries <- fallback_data %>%
                            dplyr::select(all_of(fcol)) %>% dplyr::slice(2:n()) %>% dplyr::pull()
                        fallback_mf[[fcol]] <- ts(fseries, start = c(1959, 2), frequency = 12)
                    }
                }

                fprior <- set_prior(Y = fallback_mf, n_lags = 4, n_reps = 20, block_exo = integer(0))
                fc_interval <- t(sapply(fallback_mf, Rmisc::CI, ci = 0.95))
                fprior_intervals <- fc_interval[, c("upper", "lower")]
                fmoments <- interval_to_moments(fprior_intervals)
                fprior <- update_prior(fprior, d = "intercept",
                                        prior_psi_mean = fmoments$prior_psi_mean,
                                        prior_psi_Omega = fmoments$prior_psi_Omega)
                fprior <- update_prior(fprior, n_fcst = 12)
                fmodel <- estimate_mfbvar(fprior, prior = "minn", variance = "iw")
                predict(fmodel, pred_bands = NULL) %>%
                    dplyr::filter(variable == target_variable, fcst_date == date) %>%
                    dplyr::select(fcst) %>% dplyr::pull() %>% mean()
            }, error = function(e2) {
                cat("Cat2 fallback failed at", date, lag_name, ":", conditionMessage(e2), "\n")
                NA
            })
        })

        pred_dict[pred_dict$date == date, lag_name] <- pred
    }

    if (idx %% 4 == 0) cat(idx, "/", length(dates), "done\n")
}

actuals <- test %>%
    dplyr::filter(date %in% as.Date(dates)) %>%
    dplyr::select(!!target_variable) %>%
    dplyr::pull()

dir.create("../predictions", showWarnings = FALSE)
for (lag_name in names(vintage_offsets)) {
    df_out <- data.frame(
        date = dates, actual = actuals,
        prediction = pred_dict[, lag_name]
    )
    write.csv(df_out, paste0("../predictions/bvar_", lag_name, ".csv"), row.names = FALSE)
}

panels <- list(
    "Pre-COVID"  = c("2017-01-01", "2019-12-31"),
    "COVID"      = c("2020-04-01", "2021-12-31"),
    "Post-COVID" = c("2022-01-01", "2025-12-31"),
    "Full"       = c("2017-01-01", "2025-12-31")
)
rmse <- function(a, p) sqrt(mean((a - p)^2, na.rm = TRUE))
mae  <- function(a, p) mean(abs(a - p), na.rm = TRUE)
d <- as.Date(dates)
for (pn in names(panels)) {
    rng <- panels[[pn]]
    m <- d >= rng[1] & d <= rng[2]
    cat("--- ", pn, " ---
")
    for (lag_name in names(vintage_offsets)) {
        cat("  ", lag_name, "  RMSFE=",
            format(rmse(actuals[m], pred_dict[m, lag_name]), digits = 6),
            "  MAE=", format(mae(actuals[m], pred_dict[m, lag_name]), digits = 6), "
")
    }
}

